# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_raw.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_raw.parquet')

In [4]:
X_train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence


In [5]:
X_test.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


# Machine Learning

In [6]:
models = dict(
    lgbm=load_pickle('../models/layer_one/model_lightgbm.pkl'),
    cat=load_pickle('../models/layer_one/model_catboost.pkl'),
    xgb=load_pickle('../models/layer_one/model_xgboost.pkl'),
    hist=load_pickle('../models/layer_one/model_histgradientboosting.pkl'),
    extra=load_pickle('../models/layer_one/model_extra_tree.pkl'),
    rf=load_pickle('../models/layer_one/model_random_forest.pkl')
)

## Train Dataset

In [7]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

In [8]:
X_train_stacking = pd.DataFrame({})

In [9]:
for model_name, model in models.items():
    
    print(f"Predicting {model_name}")

    predictions = cross_val_predict(model, X_train, y_train.class_encoded, cv=cv, n_jobs=-1, method='predict_proba')
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1']] = predictions[:, [0, 1]]

Predicting lgbm
Predicting cat
Predicting xgb
Predicting hist
Predicting extra
Predicting rf


## Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({})

In [11]:
for model_name, model in models.items():
    
    print(f"Predicting {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1']] = model.predict_proba(X_test)[:, [0, 1]]

Predicting lgbm
Predicting cat
Predicting xgb
Predicting hist
Predicting extra
Predicting rf


# Saving

In [12]:
X_train_stacking.to_parquet('../data/X_train_stacking_layer_one.parquet')
X_test_stacking.to_parquet('../data/X_test_stacking_layer_one.parquet')

In [13]:
X_train_stacking.head()

,lgbm_0,lgbm_1,cat_0,cat_1,xgb_0,xgb_1,hist_0,hist_1,extra_0,extra_1,rf_0,rf_1
0,9.999566e-01,0.000043,0.999900,0.000050,0.999734,0.000192,9.998896e-01,0.000105,0.912643,0.007039,0.999665,0.000155
1,9.895666e-01,0.000366,0.986604,0.000093,0.985616,0.000664,9.819315e-01,0.000384,0.803503,0.004824,0.965267,0.000184
2,3.890271e-07,1.000000,0.000016,0.999984,0.000088,0.999889,3.930889e-07,1.000000,0.006700,0.965519,0.000113,0.999887
3,9.998353e-01,0.000164,0.999925,0.000071,0.999614,0.000265,9.991001e-01,0.000895,0.932451,0.006030,0.999310,0.000345
4,9.993004e-01,0.000692,0.999187,0.000766,0.998981,0.000832,9.846569e-01,0.015320,0.915246,0.008021,0.997708,0.000696
